In [1]:
from numba import njit, prange
import numpy as np

import faiss

data = np.load("/home/dhem/workspace/2024.3/data/save/train_test-final-15063087582523.npz")

x = data["x"]
y = data["y"]
w = data["w"]
coor = data["coor"]
name = data["name"]

In [2]:
(num_sample, dim_sample) = x.shape
res = faiss.StandardGpuResources()
flat_config = faiss.GpuIndexFlatConfig()
flat_config.device = 0
index = faiss.GpuIndexFlatL2(res, dim_sample, flat_config)
index.add(x)

In [3]:
b = x.copy()
min_number = 4
distances, indices = index.search(b, min_number)

var_y = np.var(np.einsum("ij,i->ij", y[indices], x[:, 0] * w), axis=1)
argsort_ = np.argsort(var_y)[::-1][:100]
print(np.einsum("ij,i->ij", y[indices], x[:, 0] * w)[argsort_])

[[2.28326695e-06 2.28326694e-06 2.28326695e-06 2.28326695e-06]
 [2.28326695e-06 2.28326694e-06 2.28326695e-06 2.28326695e-06]
 [2.28801548e-06 2.28801549e-06 2.28801548e-06 2.28801548e-06]
 [2.28801548e-06 2.28801549e-06 2.28801548e-06 2.28801548e-06]
 [2.28326695e-06 2.28326695e-06 2.28326694e-06 2.28326694e-06]
 [2.28326695e-06 2.28326695e-06 2.28326696e-06 2.28326695e-06]
 [2.28801548e-06 2.28801548e-06 2.28801548e-06 2.28801548e-06]
 [2.28801548e-06 2.28801548e-06 2.28801548e-06 2.28801548e-06]
 [2.32047453e-06 2.32047453e-06 2.32047453e-06 2.32047453e-06]
 [2.32047454e-06 2.32047454e-06 2.32047454e-06 2.32047454e-06]
 [2.32047454e-06 2.32047453e-06 2.32047454e-06 2.32047453e-06]
 [2.30586131e-06 2.30586131e-06 2.30586131e-06 2.30586131e-06]
 [2.36524061e-06 2.36524062e-06 2.36524061e-06 2.36524061e-06]
 [2.28801548e-06 2.28801549e-06 2.28801549e-06 2.28801548e-06]
 [2.32047454e-06 2.32047454e-06 2.32047453e-06 2.32047453e-06]
 [2.13410426e-06 2.13410426e-06 2.13410425e-06 2.134104

In [4]:
distance = np.sum(
    np.transpose(
        np.transpose(x[indices[argsort_]], axes=(0, 2, 1)) - x[argsort_][:, :, None],
        axes=(0, 2, 1),
    ) ** 2,
    axis=2,
)
# print(x[argsort_][:, :, None].shape)
# print(np.transpose(x[indices[argsort_]], axes=(0, 2, 1)).shape)
energy = np.einsum("ij,i->ij", y[indices[argsort_]], (x[:, 0] * w)[argsort_])
print(y[indices[argsort_]] - y[argsort_][:, None])
print(energy)

[[ 0.00000000e+00 -1.25005897e-07  1.35747214e-07  1.00276608e-07]
 [ 0.00000000e+00 -1.24293692e-07  1.38489511e-07  9.44619956e-08]
 [ 0.00000000e+00  1.35555311e-07 -1.29841027e-07 -2.83465766e-08]
 [ 0.00000000e+00  1.35567674e-07 -1.29854840e-07 -2.57170996e-08]
 [ 0.00000000e+00  1.29696716e-07 -1.21260584e-07 -4.89644094e-08]
 [ 0.00000000e+00 -1.28976552e-07  1.20482099e-07  4.87187179e-08]
 [ 0.00000000e+00 -1.17885818e-07  1.34023836e-07  1.30100091e-08]
 [ 0.00000000e+00 -1.16561111e-07  1.34304415e-07  1.90868406e-08]
 [ 0.00000000e+00  1.29841027e-07 -2.54799915e-08 -1.14539390e-07]
 [ 0.00000000e+00 -1.29174481e-07  2.29944987e-08  1.13849580e-07]
 [ 0.00000000e+00 -1.31438824e-07  8.61137437e-08 -1.00276608e-07]
 [ 0.00000000e+00 -1.38489511e-07  9.54346575e-08  3.48237137e-08]
 [ 0.00000000e+00  1.34030330e-07 -8.38067393e-08  8.68655405e-08]
 [ 0.00000000e+00  1.31438824e-07  8.67047731e-08 -9.38436813e-08]
 [ 0.00000000e+00  8.52331681e-08 -1.32603901e-07 -9.44619956e

In [10]:
from matplotlib import pyplot as plt

color_dict = {
    "methane_cc-pVDZ_0-1_1_-0.2000": "#004D40",
    "methane_cc-pVDZ_0-1_1_-0.1000": "#1A237E",
    "methane_cc-pVDZ_0-1_1_0.0000": "#7B1FA2",
    "methane_cc-pVDZ_0-1_1_0.1000": "#B71C1C",
    "methane_cc-pVDZ_0-1_1_0.2000": "#FF6F00",
}

plt.rcParams["figure.figsize"] = np.array([3, 3]) * 520 / 72

f, axes = plt.subplots(10, 10)
axes = axes.reshape(10, 10)

begin_y = 0.025
end_y = 0.95
int_y = 0.0
begin_x = 0.025
end_x = 0.95
int_x = 0.0
end_x += int_x
end_y += int_y

shapexy = np.shape(axes)
inter_x = np.linspace(begin_x, end_x, shapexy[1] + 1)
inter_y = np.linspace(begin_y, end_y, shapexy[0] + 1)

delta_x = inter_x[1] - inter_x[0] - int_x
delta_y = inter_y[1] - inter_y[0] - int_y

for i in range(shapexy[0]):
    for j in range(shapexy[1]):
        axes[i][j].set_position(
            [
                inter_x[j],
                inter_y[i],
                inter_x[j + 1] - inter_x[j] - int_x,
                inter_y[i + 1] - inter_y[i] - int_y,
            ]
        )
        axes[i][j].xaxis.set_tick_params(
            direction="in", which="both", bottom=True, top=True
        )
        axes[i][j].yaxis.set_tick_params(
            direction="in", which="both", left=True, right=True
        )
        if i != 0:
            axes[i][j].set_xticks([])
        if j != 0:
            axes[i][j].set_yticks([])


for i in range(argsort_.shape[0]):
    axes_i, axes_j = np.unravel_index(i, (10, 10))
    for j in range(indices.shape[1]):
        axes[axes_i, axes_j].scatter(
            distance[i][j],
            np.abs(energy[i][j] - energy[i][0]) * 627.509,
            c=color_dict[name[indices[argsort_[i]]][j]],
        )
        axes[axes_i, axes_j].set_xlim(-2, 22)
        axes[axes_i, axes_j].set_ylim(-0.1, 1.1)
plt.savefig("test.pdf", dpi=300)
plt.clf()

<Figure size 2166.67x2166.67 with 0 Axes>